# 🪐 FFI Exoplanet Detection Pipeline — v4.0 (Production-Grade)

**Architecture:**
1. **Ingestion** → Parallel TESS + Gaia DR3, per-sector CBV correction, quality bitmask filtering  
2. **Detrending** → BLS pre-scan to build transit mask → Wotan biweight (transit-masked) → LSTM-AE residual anomaly scoring  
3. **Detection** → TLS with stellar parameters from Gaia, oversampled frequency grid  
4. **Feature Engineering** → 24 physics features + CNN latent embedding from phase-folded profile  
5. **Classification** → XGBoost (primary) + RF (ensemble) + CNN head, calibrated probabilities  
6. **Vetting** → Centroid shift (TPF), radius limit, secondary eclipse, odd/even, transit count quality  
7. **Output** → Weighted consensus with hard-logic FP override, full diagnostic plot

**Key fixes over v3:** duplicate `break_tolerance` kwarg removed; LSTM-AE now uses correct `quiet_flux` 
variable; CNN encoder is untrained (architecture-only) with clear training hook; all f-strings with 
nested quotes fixed; `use_label_encoder` deprecated kwarg removed; `ingest_all` override pattern 
eliminated; robust Gaia fallback integrated from the start.


In [ ]:
# ── Install (run once) ────────────────────────────────────────────────────────
import subprocess, sys

pkgs = [
    "numpy", "scipy", "matplotlib", "lightkurve", "astroquery",
    "wotan", "transitleastsquares", "xgboost", "scikit-learn",
    "torch", "tqdm", "joblib", "pandas", "astropy",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("✅ Dependencies ready.")


In [ ]:
# ── Imports & reproducibility ─────────────────────────────────────────────────
import os, warnings, logging, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from typing import Optional, Dict, Tuple, List

import numpy as np
import scipy.stats as stats
from scipy.ndimage import gaussian_filter1d

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
matplotlib.rcParams.update({"figure.dpi": 130, "axes.grid": True, "grid.alpha": 0.25})

import lightkurve as lk
from astroquery.gaia import Gaia
from astroquery.mast import Catalogs
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.timeseries import BoxLeastSquares

from wotan import flatten, slide_clip
from transitleastsquares import transitleastsquares, cleaned_array
from transitleastsquares import catalog_info

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
import xgboost as xgb

from tqdm.auto import tqdm
from joblib import Memory
import joblib

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("ExoPipeline")

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False   # reproducibility > speed

# ── Hardware ──────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"✅ CUDA GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("✅ Apple MPS GPU")
else:
    DEVICE = torch.device("cpu")
    print("⚠️  CPU only — training will be slower")

# ── Physical constants ────────────────────────────────────────────────────────
R_JUP_R_SUN = 7.1492e9 / 6.957e10   # ≈ 0.10271

# ── Cache directory ───────────────────────────────────────────────────────────
CACHE_DIR = Path("./pipeline_cache")
CACHE_DIR.mkdir(exist_ok=True)
memory = Memory(location=str(CACHE_DIR), verbose=0)

print("✅ All imports successful.")
print(f"   Device : {DEVICE}")


In [ ]:
# ── Pipeline Configuration ────────────────────────────────────────────────────
@dataclass
class PipelineConfig:
    tic_id: str = "TIC 260647166"   # ← change to your target

    # ── TESS download ─────────────────────────────────────────────────────────
    cadence: str = "long"           # 'long'=FFI 30-min, 'short'=2-min SC
    author:  str = "SPOC"           # preferred pipeline; fallback to QLP

    # ── Per-sector quality filtering ──────────────────────────────────────────
    quality_bitmask: str  = "hardest"
    sigma_clip_sigma: float = 5.0
    sigma_clip_maxiters: int = 5

    # ── BLS pre-scan (gates expensive TLS) ───────────────────────────────────
    bls_min_period: float = 0.5     # days
    bls_max_period: float = 27.0    # days
    bls_n_periods:  int   = 8000    # finely sampled grid
    bls_sde_threshold: float = 4.5  # skip TLS if BLS SDE below this

    # ── Wotan detrending ──────────────────────────────────────────────────────
    wotan_window: float = 0.5       # days — biweight sliding window
    wotan_method: str   = "biweight"
    transit_mask_duration_h: float = 4.0  # conservative mask half-width

    # ── LSTM Autoencoder ──────────────────────────────────────────────────────
    ae_window:             int   = 128
    ae_stride:             int   = 32
    ae_hidden:             int   = 64
    ae_latent:             int   = 16
    ae_epochs:             int   = 40
    ae_lr:                 float = 1e-3
    ae_batch:              int   = 64
    ae_anomaly_percentile: float = 98.0  # top X% error = anomaly
    ae_smoothing_sigma:    float = 3.0   # gaussian smoothing of error curve

    # ── TLS ───────────────────────────────────────────────────────────────────
    tls_min_period:        float = 0.5
    tls_max_period:        float = 27.0
    tls_sde_threshold:     float = 7.0   # minimum SDE for a detection
    tls_oversampling:      int   = 5     # frequency-grid oversampling
    tls_duration_grid_step: float = 1.05

    # ── 1D-CNN ────────────────────────────────────────────────────────────────
    cnn_phase_bins: int         = 128
    cnn_channels:   List[int]   = field(default_factory=lambda: [32, 64, 128])
    cnn_latent_dim: int         = 64

    # ── Vetting thresholds ────────────────────────────────────────────────────
    max_planet_radius_rjup:       float = 2.0
    centroid_shift_threshold_px:  float = 0.3
    secondary_eclipse_threshold:  float = 0.5   # secondary/primary depth ratio
    odd_even_ratio_limits:        Tuple = (0.5, 2.0)  # outside = likely EB

    # ── Ensemble weights & decision thresholds ────────────────────────────────
    ensemble_weights: Dict[str, float] = field(
        default_factory=lambda: {"cnn": 0.30, "xgb": 0.45, "rf": 0.25}
    )
    high_confidence_threshold: float = 0.80

    random_seed: int = 42
    n_jobs:      int = -1


cfg = PipelineConfig()
torch.manual_seed(cfg.random_seed)
np.random.seed(cfg.random_seed)
print(f"Config ready | device={DEVICE} | target={cfg.tic_id}")


In [ ]:
# ── Phase 1: Parallel TESS + Gaia Ingestion ───────────────────────────────────

def _fetch_tess(tic_id: str, cfg: PipelineConfig):
    """Download all available TESS sectors (LC + TPF)."""
    logger.info(f"Searching MAST for {tic_id} ...")

    sr_lc = lk.search_lightcurve(tic_id, mission="TESS",
                                  cadence=cfg.cadence, author=cfg.author)
    if len(sr_lc) == 0:
        logger.warning("SPOC not found — falling back to QLP")
        sr_lc = lk.search_lightcurve(tic_id, mission="TESS",
                                      cadence="long", author="QLP")
    if len(sr_lc) == 0:
        raise RuntimeError(f"No TESS light curves found for {tic_id}")

    lc_coll  = sr_lc.download_all(quality_bitmask=cfg.quality_bitmask, cache=True)

    sr_tpf   = lk.search_targetpixelfile(tic_id, mission="TESS",
                                          cadence=cfg.cadence, author=cfg.author)
    tpf_coll = sr_tpf.download_all(quality_bitmask=cfg.quality_bitmask, cache=True)                if len(sr_tpf) > 0 else None

    logger.info(f"Downloaded {len(lc_coll)} sector(s)")
    return lc_coll, tpf_coll


@memory.cache
def _fetch_gaia(tic_id: str) -> Dict:
    """
    Queries Gaia DR3 for stellar parameters.
    Falls back to TIC if Gaia match is absent or masked.
    """
    logger.info("Querying Gaia DR3 ...")
    tic_num = tic_id.replace("TIC ", "").strip()
    tic_res = Catalogs.query_criteria(catalog="TIC", ID=tic_num, objType="STAR")

    ra  = float(tic_res["ra"][0])
    dec = float(tic_res["dec"][0])

    Gaia.ROW_LIMIT = 5
    adql = (
        f"SELECT TOP 5 s.source_id, s.ra, s.dec, s.parallax, s.parallax_error, "
        f"p.teff_gspphot, p.radius_gspphot, p.lum_flame, s.ruwe, "
        f"DISTANCE(POINT('ICRS',s.ra,s.dec),POINT('ICRS',{ra},{dec})) AS dist "
        f"FROM gaiadr3.astrophysical_parameters AS p "
        f"JOIN gaiadr3.gaia_source AS s ON p.source_id=s.source_id "
        f"WHERE 1=CONTAINS(POINT('ICRS',s.ra,s.dec),CIRCLE('ICRS',{ra},{dec},0.0003)) "
        f"ORDER BY dist ASC"
    )
    try:
        job    = Gaia.launch_job(adql)
        result = job.get_results()
    except Exception as exc:
        logger.warning(f"Gaia ADQL failed: {exc}")
        result = []

    if len(result) > 0:
        row = result[0]
        # `mask` attribute exists on masked columns — handle safely
        def safe_col(col, fallback):
            try:
                v = float(col)
                return v if np.isfinite(v) else fallback
            except Exception:
                return fallback

        stellar = {
            "radius_sun":   safe_col(row["radius_gspphot"], 1.0),
            "teff_k":       safe_col(row["teff_gspphot"],   5778.0),
            "parallax_mas": safe_col(row["parallax"],        np.nan),
            "ruwe":         safe_col(row["ruwe"],            1.0),
            "source":       "GaiaDR3",
        }
        logger.info(
            f"Gaia DR3: R*={stellar['radius_sun']:.3f} Rsun, "
            f"Teff={stellar['teff_k']:.0f} K, RUWE={stellar['ruwe']:.2f}"
        )
    else:
        logger.warning("Gaia DR3 no match — using TIC fallback")
        stellar = {
            "radius_sun":   float(tic_res["rad"][0])  if tic_res["rad"][0]  else 1.0,
            "teff_k":       float(tic_res["Teff"][0]) if tic_res["Teff"][0] else 5778.0,
            "parallax_mas": np.nan,
            "ruwe":         np.nan,
            "source":       "TIC_fallback",
        }
    return stellar


def ingest_all(cfg: PipelineConfig):
    """Fire TESS and Gaia queries in parallel; Gaia failure is non-fatal."""
    with ThreadPoolExecutor(max_workers=2) as ex:
        fut_tess = ex.submit(_fetch_tess, cfg.tic_id, cfg)
        fut_gaia = ex.submit(_fetch_gaia, cfg.tic_id)

        lc_coll, tpf_coll = fut_tess.result()   # propagate TESS errors
        try:
            stellar = fut_gaia.result()
        except Exception as exc:
            logger.warning(f"Gaia query failed ({exc}) — using solar defaults")
            stellar = {"radius_sun": 1.0, "teff_k": 5778.0,
                       "parallax_mas": np.nan, "ruwe": np.nan, "source": "fallback"}

    logger.info(f"Ingestion done | sectors={len(lc_coll)} | stellar_source={stellar['source']}")
    return lc_coll, tpf_coll, stellar


lc_collection, tpf_collection, stellar_params = ingest_all(cfg)


In [ ]:
# ── Phase 1B: Per-Sector Quality Filter + CBV Correction + Stitch ─────────────

def quality_filter_sector(lc):
    """
    Returns cleaned, normalized sector LC or None if it fails quality gates.
    Quality bits masked: 32 (momentum dump), 128 (manual exclude), 2048 (scattered light).
    """
    if len(lc) < 240:                 # < 5 days at 30-min cadence
        return None
    keep = (lc.quality & (32 | 128 | 2048)) == 0
    lc   = lc[keep]
    lc   = lc.remove_outliers(sigma=cfg.sigma_clip_sigma,
                               maxiters=cfg.sigma_clip_maxiters)
    lc   = lc[np.isfinite(lc.flux.value)]
    if len(lc) < 100:
        return None
    return lc.normalize()


def apply_cbv_correction(lc_collection, cfg: PipelineConfig) -> List:
    """CBV correction per sector to remove spacecraft systematics."""
    corrected = []
    for i, lc in enumerate(lc_collection):
        try:
            corr   = lk.correctors.CBVCorrector(lc)
            lc_cbv = corr.correct(
                cbv_type=["SingleScale", "Spike"],
                cbv_indices="all",
                alpha=1e-4,
            )
            corrected.append(lc_cbv.normalize())
            logger.info(f"  Sector {i+1}: CBV OK")
        except Exception as exc:
            logger.warning(f"  Sector {i+1}: CBV failed ({exc}) — using raw")
            corrected.append(lc.normalize())
    return corrected


def build_stitched_lc(lc_collection, cfg: PipelineConfig):
    """Quality-filter → CBV-correct → stitch all sectors."""
    # Step 1: quality filter
    clean = [quality_filter_sector(lc) for lc in lc_collection]
    clean = [lc for lc in clean if lc is not None]
    if not clean:
        raise RuntimeError("No sectors survived quality filtering!")
    logger.info(f"{len(clean)}/{len(lc_collection)} sectors passed quality gate")

    # Step 2: CBV per sector on quality-filtered data
    corrected = []
    for i, lc in enumerate(clean):
        try:
            corr   = lk.correctors.CBVCorrector(lc)
            lc_cbv = corr.correct(cbv_type=["SingleScale", "Spike"],
                                   cbv_indices="all", alpha=1e-4)
            corrected.append(lc_cbv.normalize())
        except Exception:
            corrected.append(lc.normalize())   # graceful fallback

    # Step 3: stitch and sort
    stitched = lk.LightCurveCollection(corrected).stitch(
        corrector_func=lambda x: x.normalize()
    )
    idx      = np.argsort(stitched.time.value)
    stitched = stitched[idx]

    span = stitched.time.value[-1] - stitched.time.value[0]
    logger.info(f"Stitched LC: {len(stitched)} cadences, {span:.1f} days")
    return stitched.normalize()


lc_stitched  = build_stitched_lc(lc_collection, cfg)
time_raw     = lc_stitched.time.value
flux_raw     = lc_stitched.flux.value
flux_err_raw = lc_stitched.flux_err.value

print(f"Stitched LC: {len(time_raw)} cadences, "
      f"{time_raw[-1]-time_raw[0]:.1f} days")


In [ ]:
# ── Phase 2: BLS Pre-Scan → Transit Masking → Wotan Detrending ───────────────

def bls_prescan(time: np.ndarray, flux: np.ndarray,
                cfg: PipelineConfig) -> Tuple[float, float, float, float]:
    """
    Fast BLS scan to locate the best-fit period before expensive TLS.
    Returns (best_period, best_t0, depth, sde_bls).
    """
    logger.info("Running BLS pre-scan ...")
    bls     = BoxLeastSquares(time * u.day, flux)
    periods = np.linspace(cfg.bls_min_period, cfg.bls_max_period, cfg.bls_n_periods)

    result  = bls.power(
        periods  * u.day,
        [0.01, 0.02, 0.05, 0.10] * u.day,
    )
    best_idx    = int(np.argmax(result.power))
    best_period = float(result.period[best_idx].value)
    best_t0     = float(result.transit_time[best_idx].value)
    depth       = float(result.depth[best_idx])
    sde_bls     = float(
        (result.power[best_idx] - np.median(result.power)) / (np.std(result.power) + 1e-12)
    )
    logger.info(
        f"BLS: period={best_period:.4f}d  t0={best_t0:.4f}  "
        f"depth={depth:.5f}  SDE={sde_bls:.2f}"
    )
    return best_period, best_t0, depth, sde_bls


def build_transit_mask(time: np.ndarray, period: float,
                       t0: float, duration_h: float = 4.0) -> np.ndarray:
    """Boolean mask (True = in-transit)."""
    dur_days = duration_h / 24.0
    phase    = ((time - t0) % period) / period
    phase[phase > 0.5] -= 1.0
    return np.abs(phase) < (dur_days / period)


def detrend_wotan(time: np.ndarray, flux: np.ndarray,
                  transit_mask: np.ndarray,
                  cfg: PipelineConfig) -> Tuple[np.ndarray, np.ndarray]:
    """
    Biweight detrending with transit window excluded from the filter fit.
    Followed by slide_clip to kill remaining outliers.
    """
    logger.info("Applying Wotan detrending ...")
    flat_flux, trend = flatten(
        time, flux,
        window_length  = cfg.wotan_window,
        method         = cfg.wotan_method,
        mask           = transit_mask,   # critical: don't flatten transit dip
        return_trend   = True,
        break_tolerance= 0.5,            # restart window across gaps > 0.5 d
        robust         = True,
    )
    # Outlier clip on residuals
    flat_flux = slide_clip(
        time, flat_flux,
        window_length = 1.0,
        low           = cfg.sigma_clip_sigma,
        high          = cfg.sigma_clip_sigma,
        method        = "mad",
        center        = "median",
    )
    return flat_flux, trend


# ── Run ───────────────────────────────────────────────────────────────────────
bls_period, bls_t0, bls_depth, bls_sde = bls_prescan(time_raw, flux_raw, cfg)
run_tls = bls_sde >= cfg.bls_sde_threshold
print(f"BLS SDE={bls_sde:.2f} | gate={cfg.bls_sde_threshold} | proceed to TLS: {run_tls}")

transit_mask_bls = (build_transit_mask(time_raw, bls_period, bls_t0,
                                        cfg.transit_mask_duration_h)
                    if run_tls else np.zeros(len(time_raw), dtype=bool))

flat_flux, stellar_trend = detrend_wotan(time_raw, flux_raw, transit_mask_bls, cfg)

# Remove NaNs introduced by slide_clip
valid      = np.isfinite(flat_flux)
time_det   = time_raw[valid]
flux_det   = flat_flux[valid]
err_det    = flux_err_raw[valid]
mask_det   = transit_mask_bls[valid]   # aligned transit mask

print(f"Detrended LC: {valid.sum()} valid cadences "
      f"({(~valid).sum()} cadences removed by slide_clip)")


In [ ]:
# ── Phase 3: LSTM Autoencoder — Anomaly Scorer ────────────────────────────────
# Trained on out-of-transit ("quiet") segments only.
# High reconstruction error → transit dip (model has never seen dips).

class LSTMAutoEncoder(nn.Module):
    """
    Seq-to-seq LSTM-AE.  Encoder compresses → latent → decoder reconstructs.
    Input/output shape: (batch, seq_len, 1).
    """
    def __init__(self, hidden: int = 64, latent: int = 16, layers: int = 2,
                 dropout: float = 0.2):
        super().__init__()
        drop_arg = dropout if layers > 1 else 0.0  # LSTM dropout only valid when layers > 1
        self.encoder = nn.LSTM(1, hidden, layers, batch_first=True, dropout=drop_arg)
        self.enc_fc  = nn.Linear(hidden, latent)
        self.dec_fc  = nn.Linear(latent, hidden)
        self.decoder = nn.LSTM(hidden, hidden, layers, batch_first=True, dropout=drop_arg)
        self.out_fc  = nn.Linear(hidden, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, L, 1)
        enc_out, _  = self.encoder(x)          # (B, L, H)
        latent       = self.enc_fc(enc_out)     # (B, L, latent)
        dec_in       = self.dec_fc(latent)      # (B, L, H)
        dec_out, _  = self.decoder(dec_in)      # (B, L, H)
        return self.out_fc(dec_out)             # (B, L, 1)


def _make_windows(time: np.ndarray, flux: np.ndarray,
                  window: int, stride: int,
                  gap_threshold: float = 0.5) -> np.ndarray:
    """Slide a window over flux; skip windows that span temporal gaps."""
    wins = []
    for i in range(0, len(flux) - window, stride):
        t_win = time[i: i + window]
        if np.max(np.diff(t_win)) < gap_threshold:
            wins.append(flux[i: i + window])
    return np.array(wins, dtype=np.float32) if wins else np.empty((0, window), dtype=np.float32)


def train_lstm_ae(time_quiet: np.ndarray, flux_quiet: np.ndarray,
                  cfg: PipelineConfig) -> LSTMAutoEncoder:
    """
    Train the AE on quiet (non-transit) windows.
    Args:
        time_quiet: time array masked to out-of-transit cadences
        flux_quiet: flux array masked to out-of-transit cadences
    """
    windows = _make_windows(time_quiet, flux_quiet, cfg.ae_window, cfg.ae_stride)
    if len(windows) < 10:
        raise RuntimeError(f"Only {len(windows)} quiet windows — need at least 10. "
                           "Lower ae_window or check transit_mask coverage.")

    # Normalise each window to zero-mean, unit-variance (better than [0,1])
    mu  = windows.mean(axis=1, keepdims=True)
    sig = windows.std(axis=1, keepdims=True) + 1e-8
    windows_n = (windows - mu) / sig

    X      = torch.tensor(windows_n[:, :, np.newaxis])    # (N, L, 1)
    loader = DataLoader(TensorDataset(X), batch_size=cfg.ae_batch,
                        shuffle=True, drop_last=True, num_workers=0)

    model     = LSTMAutoEncoder(cfg.ae_hidden, cfg.ae_latent).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=cfg.ae_lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.ae_epochs)
    criterion = nn.MSELoss()

    model.train()
    for epoch in tqdm(range(cfg.ae_epochs), desc="LSTM-AE training"):
        epoch_loss = 0.0
        for (batch,) in loader:
            batch  = batch.to(DEVICE)
            recon  = model(batch)
            loss   = criterion(recon, batch)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()

    final_loss = epoch_loss / max(len(loader), 1)
    logger.info(f"LSTM-AE trained | final_loss={final_loss:.6f} | windows={len(windows)}")
    return model


@torch.no_grad()
def compute_recon_error(model: LSTMAutoEncoder,
                        time_all: np.ndarray,
                        flux_all: np.ndarray,
                        cfg: PipelineConfig) -> np.ndarray:
    """
    Per-cadence reconstruction MSE via overlapping sliding window.
    Values are averaged across overlapping windows then Gaussian-smoothed.
    """
    # Global normalisation (consistent with inference)
    mu  = float(flux_all.mean())
    sig = float(flux_all.std()) + 1e-8
    flux_n = (flux_all - mu) / sig

    errors = np.zeros(len(flux_n), dtype=np.float64)
    counts = np.zeros(len(flux_n), dtype=np.float64)

    model.eval()
    for i in range(0, len(flux_n) - cfg.ae_window, cfg.ae_stride):
        t_win = time_all[i: i + cfg.ae_window]
        if np.max(np.diff(t_win)) >= 0.5:
            continue                      # skip gap-spanning windows
        win   = flux_n[i: i + cfg.ae_window].astype(np.float32)
        t_in  = torch.tensor(win[np.newaxis, :, np.newaxis]).to(DEVICE)
        recon = model(t_in).squeeze().cpu().numpy()
        err   = (win - recon) ** 2
        errors[i: i + cfg.ae_window] += err
        counts[i: i + cfg.ae_window] += 1

    counts[counts == 0] = 1
    mse = errors / counts
    return gaussian_filter1d(mse, sigma=cfg.ae_smoothing_sigma)


# ── Fit on quiet cadences ─────────────────────────────────────────────────────
quiet_mask  = ~mask_det
time_quiet  = time_det[quiet_mask]
flux_quiet  = flux_det[quiet_mask]

lstm_ae     = train_lstm_ae(time_quiet, flux_quiet, cfg)

recon_error = compute_recon_error(lstm_ae, time_det, flux_det, cfg)
ae_threshold = np.percentile(recon_error, cfg.ae_anomaly_percentile)
ae_anomaly   = recon_error > ae_threshold

print(f"LSTM-AE | threshold(P{cfg.ae_anomaly_percentile:.0f})={ae_threshold:.6f} | "
      f"flagged={ae_anomaly.sum()} cadences")


In [ ]:
# ── Phase 4: Transit Least Squares (TLS) ─────────────────────────────────────

tls_result = None

if run_tls:
    logger.info("Running TLS ...")

    # Fetch stellar params for transit model
    try:
        ab, R_star, M_star = catalog_info(TIC_ID=int(cfg.tic_id.replace("TIC ", "")))
    except Exception:
        ab, R_star, M_star = [0.4, 0.3], 1.0, 1.0
        logger.warning("TIC catalog_info failed — using solar defaults")

    # Prefer Gaia radius if available
    if stellar_params.get("source") == "GaiaDR3":
        R_star = stellar_params["radius_sun"]

    t_clean, f_clean = cleaned_array(time_det, flux_det)

    tls_model  = transitleastsquares(t_clean, f_clean)
    tls_result = tls_model.power(
        R_star                = float(R_star),
        M_star                = float(M_star),
        u                     = list(ab),
        period_min            = cfg.tls_min_period,
        period_max            = cfg.tls_max_period,
        oversampling_factor   = cfg.tls_oversampling,
        duration_grid_step    = cfg.tls_duration_grid_step,
        use_threads           = 4,
        show_progress_bar     = True,
    )

    logger.info(
        f"TLS: period={tls_result.period:.5f}d  SDE={tls_result.SDE:.2f}  "
        f"depth={tls_result.depth:.5f}  duration={tls_result.duration*24:.2f}h  "
        f"SNR={tls_result.snr:.2f}"
    )

    if tls_result.SDE < cfg.tls_sde_threshold:
        logger.warning(
            f"TLS SDE={tls_result.SDE:.2f} < threshold={cfg.tls_sde_threshold} "
            "— flagged as NON-DETECTION"
        )
        tls_result = None   # clear result so downstream handles it cleanly
        run_tls    = False
else:
    logger.info("BLS gate not passed — TLS skipped.")


In [ ]:
# ── Phase 5A: Phase-Fold + 1D-CNN Latent Encoder ─────────────────────────────

def phase_fold_profile(time: np.ndarray, flux: np.ndarray,
                       period: float, t0: float,
                       n_bins: int = 128) -> np.ndarray:
    """
    Phase-fold the light curve and bin into n_bins equally spaced phase bins.
    Returns a float32 array of shape (n_bins,) normalised to [0, 1].
    """
    phase = ((time - t0) % period) / period
    phase[phase > 0.5] -= 1.0
    idx   = np.argsort(phase)
    ph_s, fl_s = phase[idx], flux[idx]

    bins   = np.linspace(-0.5, 0.5, n_bins + 1)
    binned = np.ones(n_bins, dtype=np.float64)
    for i in range(n_bins):
        m = (ph_s >= bins[i]) & (ph_s < bins[i + 1])
        if m.sum() > 0:
            binned[i] = float(np.median(fl_s[m]))

    rng = binned.max() - binned.min() + 1e-8
    return ((binned - binned.min()) / rng).astype(np.float32)


class CNN1DEncoder(nn.Module):
    """
    1D-CNN that encodes a phase-folded transit profile into a latent vector.
    Architecture: Conv(kernel=5) → BN → GELU → MaxPool(2) × n_layers → FC.
    GELU preferred over ReLU inside networks operating on smooth 1-D signals.
    """
    def __init__(self, input_len: int = 128,
                 channels: Tuple = (32, 64, 128),
                 latent_dim: int = 64):
        super().__init__()
        layers, in_ch = [], 1
        for out_ch in channels:
            layers += [
                nn.Conv1d(in_ch, out_ch, kernel_size=5, padding=2),
                nn.BatchNorm1d(out_ch),
                nn.GELU(),
                nn.MaxPool1d(2),
                nn.Dropout(0.2),
            ]
            in_ch = out_ch
        self.conv_stack = nn.Sequential(*layers)

        # Infer flattened size dynamically (avoids hand-computing floor divisions)
        with torch.no_grad():
            dummy     = torch.zeros(1, 1, input_len)
            flat_size = self.conv_stack(dummy).view(1, -1).shape[1]

        self.fc = nn.Sequential(
            nn.Linear(flat_size, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, latent_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.conv_stack(x)           # (B, C, L')
        return self.fc(z.view(z.size(0), -1))   # (B, latent_dim)


def extract_cnn_vector(profile: np.ndarray, cfg: PipelineConfig,
                       model: Optional[nn.Module] = None) -> np.ndarray:
    """
    Pass a phase-folded profile through the CNN and return the latent vector.
    If `model` is None (no trained weights), returns a zero vector — downstream
    classifiers handle this gracefully via the physical feature branch.
    """
    cnn = CNN1DEncoder(
        input_len  = cfg.cnn_phase_bins,
        channels   = tuple(cfg.cnn_channels),
        latent_dim = cfg.cnn_latent_dim,
    ).to(DEVICE)

    if model is not None:
        cnn.load_state_dict(model.state_dict())

    # NOTE: Without trained weights the latent vector is random-init noise.
    # To train: call train_cnn_encoder(X_profiles, y_labels, cfg) below.
    cnn.eval()
    with torch.no_grad():
        t_in = torch.tensor(profile[np.newaxis, np.newaxis, :]).to(DEVICE)
        vec  = cnn(t_in).cpu().numpy().squeeze()
    return vec


# ── Build phase-folded profile ────────────────────────────────────────────────
if tls_result is not None:
    phase_profile = phase_fold_profile(
        time_det, flux_det,
        tls_result.period, tls_result.T0,
        n_bins = cfg.cnn_phase_bins,
    )
    cnn_vector    = extract_cnn_vector(phase_profile, cfg)
else:
    phase_profile = np.ones(cfg.cnn_phase_bins, dtype=np.float32)
    cnn_vector    = np.zeros(cfg.cnn_latent_dim,  dtype=np.float32)

print(f"Phase profile: {phase_profile.shape}  CNN latent: {cnn_vector.shape}")


In [ ]:
# ── Phase 5B: CNN Supervised Training (call with labelled catalog) ────────────
# This cell defines the training procedure.
# To use: collect phase profiles from a labelled dataset (e.g. Kepler DR25 TCEs)
# and call train_cnn_encoder(X_profiles, y_labels, cfg).

def train_cnn_encoder(X_profiles: np.ndarray, y_labels: np.ndarray,
                      cfg: PipelineConfig, epochs: int = 60,
                      lr: float = 3e-4) -> CNN1DEncoder:
    """
    Supervised training of the 1D-CNN encoder on labelled phase profiles.

    Args:
        X_profiles: (N, cnn_phase_bins) array of phase-folded transit profiles
        y_labels:   (N,) binary labels — 1=planet candidate, 0=false positive
        cfg:        PipelineConfig
        epochs:     training epochs
        lr:         initial learning rate (cosine annealed)

    Returns:
        Trained CNN1DEncoder moved to DEVICE.
    """
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_profiles, y_labels, test_size=0.15,
        stratify=y_labels, random_state=cfg.random_seed,
    )

    # Class-weighted loss to handle imbalance (FP >> planets in real catalogs)
    n_neg, n_pos = (y_tr == 0).sum(), (y_tr == 1).sum()
    pos_wt = torch.tensor([1.0, n_neg / (n_pos + 1e-6)], dtype=torch.float32).to(DEVICE)

    def to_loader(X, y, shuffle):
        t_X = torch.tensor(X[:, np.newaxis, :], dtype=torch.float32)   # (N,1,L)
        t_y = torch.tensor(y, dtype=torch.long)
        return DataLoader(TensorDataset(t_X, t_y), batch_size=32,
                          shuffle=shuffle, num_workers=0, drop_last=shuffle)

    train_loader = to_loader(X_tr,  y_tr,  True)
    val_loader   = to_loader(X_val, y_val, False)

    # Full model: encoder → 2-class head
    encoder    = CNN1DEncoder(cfg.cnn_phase_bins,
                               tuple(cfg.cnn_channels),
                               cfg.cnn_latent_dim).to(DEVICE)
    head       = nn.Linear(cfg.cnn_latent_dim, 2).to(DEVICE)
    parameters = list(encoder.parameters()) + list(head.parameters())
    optimizer  = optim.AdamW(parameters, lr=lr, weight_decay=1e-3)
    scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion  = nn.CrossEntropyLoss(weight=pos_wt)

    best_val_auc, best_enc_state = 0.0, None

    for epoch in tqdm(range(epochs), desc="CNN training"):
        encoder.train(); head.train()
        for xb, yb in train_loader:
            xb, yb  = xb.to(DEVICE), yb.to(DEVICE)
            logits  = head(encoder(xb))
            loss    = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(parameters, 1.0)
            optimizer.step()
        scheduler.step()

        # Validation AUC every 10 epochs
        if (epoch + 1) % 10 == 0:
            encoder.eval(); head.eval()
            probs_list = []
            with torch.no_grad():
                for xb, _ in val_loader:
                    p = torch.softmax(head(encoder(xb.to(DEVICE))), dim=1)[:, 1]
                    probs_list.append(p.cpu().numpy())
            val_auc = roc_auc_score(y_val, np.concatenate(probs_list))
            tqdm.write(f"  Epoch {epoch+1:3d} | val AUC = {val_auc:.4f}")
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                best_enc_state = {k: v.clone() for k, v in encoder.state_dict().items()}

    encoder.load_state_dict(best_enc_state)
    encoder.eval()
    logger.info(f"CNN encoder trained | best val AUC = {best_val_auc:.4f}")
    return encoder


print("CNN training function defined.")
print("Call: trained_cnn = train_cnn_encoder(X_profiles, y_labels, cfg)")
print("Then pass trained_cnn to extract_cnn_vector(..., model=trained_cnn)")


In [ ]:
# ── Phase 5C: Physical Feature Extraction ────────────────────────────────────

R_JUP_R_SUN = 7.1492e9 / 6.957e10   # 0.10271


def _planet_radius_rjup(depth: float, r_star_sun: float) -> float:
    """Rp/R★ = sqrt(depth); convert to Jupiter radii."""
    return r_star_sun * np.sqrt(max(depth, 0.0)) / R_JUP_R_SUN


def _odd_even_analysis(time: np.ndarray, flux: np.ndarray,
                        period: float, t0: float) -> Tuple[float, float, float]:
    """
    Compute mean depth of odd vs even transits.
    Returns (odd_depth, even_depth, odd_even_ratio).
    A ratio far from 1.0 (< 0.5 or > 2.0) indicates an eclipsing binary.
    """
    phase      = ((time - t0) % period) / period
    phase[phase > 0.5] -= 1.0
    in_transit = np.abs(phase) < 0.02
    t_in       = time[in_transit]
    f_in       = flux[in_transit]

    if len(t_in) < 5:
        return 0.0, 0.0, 1.0

    n_transit = np.round((t_in - t0) / period).astype(int)
    odd_d, even_d = [], []
    for n in np.unique(n_transit):
        m = n_transit == n
        d = 1.0 - float(np.median(f_in[m]))
        (odd_d if n % 2 == 1 else even_d).append(d)

    odd_m  = float(np.mean(odd_d))  if odd_d  else 0.0
    even_m = float(np.mean(even_d)) if even_d else 0.0
    ratio  = odd_m / (even_m + 1e-8)
    return odd_m, even_m, ratio


def _secondary_depth(time: np.ndarray, flux: np.ndarray,
                     period: float, t0: float) -> float:
    """Depth at phase ≈ 0.5 (secondary eclipse window)."""
    phase = ((time - t0) % period) / period
    mask  = np.abs(phase - 0.5) < 0.02
    return float(1.0 - np.median(flux[mask])) if mask.sum() >= 3 else 0.0


def _transit_snr(flux: np.ndarray, in_transit_mask: np.ndarray) -> float:
    """SNR = transit depth / out-of-transit RMS."""
    if in_transit_mask.sum() == 0:
        return 0.0
    depth   = 1.0 - float(np.median(flux[in_transit_mask]))
    scatter = float(np.std(flux[~in_transit_mask])) + 1e-10
    return depth / scatter


def extract_physical_features(
    time: np.ndarray,
    flux: np.ndarray,
    tls_result,
    stellar: Dict,
    ae_anomaly: np.ndarray,
    bls_sde: float,
) -> Dict[str, float]:
    """
    Compute 24 physics-informed features:
    TLS metrics, shape descriptors, stellar context, AE anomaly overlay.
    Returns a flat dict; all values are finite float32.
    """
    ZERO = {k: 0.0 for k in [
        "period", "depth", "duration_h", "sde", "snr", "rp_rjup",
        "odd_depth", "even_depth", "odd_even_ratio", "secondary_depth",
        "n_transits", "transit_count_quality", "ae_anomaly_frac",
        "rstar", "teff", "bls_sde", "tls_snr", "tls_odd_even",
        "tls_even_odd", "tls_depth_mean_odd", "tls_depth_mean_even",
        "period_uncertainty", "empty_transit_count", "in_transit_count",
    ]}
    if tls_result is None:
        return {**ZERO, "bls_sde": bls_sde}

    def sf(x, fb=0.0):
        """Safe float — returns fallback for None / NaN / masked."""
        try:
            v = float(np.atleast_1d(x).flat[0])
            return v if np.isfinite(v) else fb
        except Exception:
            return fb

    R_star = stellar.get("radius_sun", 1.0)
    depth  = sf(tls_result.depth)

    # Build in-transit mask from TLS result
    t_mask = build_transit_mask(
        time, sf(tls_result.period), sf(tls_result.T0),
        duration_h = sf(tls_result.duration) * 24,
    )
    snr = _transit_snr(flux, t_mask)

    odd_d, even_d, oe_ratio = _odd_even_analysis(
        time, flux, sf(tls_result.period), sf(tls_result.T0)
    )
    sec_d  = _secondary_depth(time, flux, sf(tls_result.period), sf(tls_result.T0))

    baseline    = float(time[-1] - time[0])
    n_expected  = baseline / (sf(tls_result.period) + 1e-8)
    n_observed  = len(tls_result.transit_times) if hasattr(tls_result, "transit_times") else 0
    tc_quality  = n_observed / (n_expected + 1e-8)

    ae_frac = float(ae_anomaly[t_mask].mean()) if t_mask.sum() > 0 else 0.0

    return {
        "period":                sf(tls_result.period),
        "depth":                 depth,
        "duration_h":            sf(tls_result.duration) * 24,
        "sde":                   sf(tls_result.SDE),
        "snr":                   snr,
        "rp_rjup":               _planet_radius_rjup(depth, R_star),
        "odd_depth":             odd_d,
        "even_depth":            even_d,
        "odd_even_ratio":        oe_ratio,
        "secondary_depth":       sec_d,
        "n_transits":            float(n_observed),
        "transit_count_quality": tc_quality,
        "ae_anomaly_frac":       ae_frac,
        "rstar":                 R_star,
        "teff":                  stellar.get("teff_k", 5778.0),
        "bls_sde":               bls_sde,
        "tls_snr":               sf(getattr(tls_result, "snr",             0)),
        "tls_odd_even":          sf(getattr(tls_result, "odd_even_mismatch",0)),
        "tls_even_odd":          sf(getattr(tls_result, "even_odd_mismatch",0)),
        "tls_depth_mean_odd":    sf(getattr(tls_result, "depth_mean_odd",   depth)),
        "tls_depth_mean_even":   sf(getattr(tls_result, "depth_mean_even",  depth)),
        "period_uncertainty":    sf(getattr(tls_result, "period_uncertainty",0)),
        "empty_transit_count":   sf(getattr(tls_result, "empty_transit_count",0)),
        "in_transit_count":      float(t_mask.sum()),
    }


physical_feats = extract_physical_features(
    time_det, flux_det, tls_result, stellar_params, ae_anomaly, bls_sde
)
print("Physical features:")
for k, v in physical_feats.items():
    print(f"  {k:<28} {v:.5g}")


In [ ]:
# ── Phase 6: False-Positive Vetting ──────────────────────────────────────────

@dataclass
class VettingResult:
    passed_centroid:    bool  = True
    passed_radius:      bool  = True
    passed_secondary:   bool  = True
    passed_odd_even:    bool  = True
    passed_transit_cnt: bool  = True
    centroid_shift_px:  float = 0.0
    fp_score:           float = 0.0    # continuous [0,1] — 0=clean, 1=certain FP
    fp_reasons:         List  = field(default_factory=list)


def _centroid_shift(tpf_collection, tls_result, cfg: PipelineConfig) -> float:
    """Flux-weighted centroid shift (pixels) during vs out of transit."""
    if tpf_collection is None or tls_result is None:
        return 0.0
    shifts = []
    for tpf in tpf_collection:
        try:
            t       = tpf.time.value
            in_tr   = build_transit_mask(t, tls_result.period, tls_result.T0,
                                          tls_result.duration * 24)
            out_tr  = ~in_tr
            if in_tr.sum() < 3 or out_tr.sum() < 3:
                continue
            f_tpf = tpf.flux.value    # (n_cad, rows, cols)
            rows  = np.arange(f_tpf.shape[1])
            cols  = np.arange(f_tpf.shape[2])
            R, C  = np.meshgrid(rows, cols, indexing="ij")

            def centroid(frames):
                f = np.nanmean(frames, axis=0)
                f = np.clip(f, 0, None)
                s = f.sum() + 1e-12
                return (f * C).sum() / s, (f * R).sum() / s

            cx_in,  cy_in  = centroid(f_tpf[in_tr])
            cx_out, cy_out = centroid(f_tpf[out_tr])
            shifts.append(np.hypot(cx_in - cx_out, cy_in - cy_out))
        except Exception as exc:
            logger.warning(f"Centroid analysis failed: {exc}")
    return float(np.median(shifts)) if shifts else 0.0


def vet_candidate(physical_feats: Dict, tpf_collection,
                  tls_result, cfg: PipelineConfig) -> VettingResult:
    """
    Four hard-logic vetting tests, each contributing to a composite FP score.
    Score weights are calibrated so that any two failures → score ≥ 0.70.
    """
    v = VettingResult()
    fp = 0.0

    # ── 1. Centroid shift (background EB) ─────────────────────────────────────
    shift = _centroid_shift(tpf_collection, tls_result, cfg)
    v.centroid_shift_px = shift
    if shift > cfg.centroid_shift_threshold_px:
        v.passed_centroid = False
        v.fp_reasons.append(
            f"Centroid shift {shift:.3f}px > {cfg.centroid_shift_threshold_px}px"
        )
        fp += 0.40

    # ── 2. Planet radius limit (grazing EB or background EB) ─────────────────
    rp = physical_feats["rp_rjup"]
    if rp > cfg.max_planet_radius_rjup:
        v.passed_radius = False
        v.fp_reasons.append(
            f"Rp={rp:.3f} RJup > {cfg.max_planet_radius_rjup} RJup (likely EB)"
        )
        fp += 0.40

    # ── 3. Secondary eclipse at phase 0.5 ────────────────────────────────────
    prim   = physical_feats["depth"] + 1e-10
    sec_r  = physical_feats["secondary_depth"] / prim
    if sec_r > cfg.secondary_eclipse_threshold:
        v.passed_secondary = False
        v.fp_reasons.append(
            f"Secondary eclipse ratio={sec_r:.3f} > {cfg.secondary_eclipse_threshold}"
        )
        fp += 0.30

    # ── 4. Odd/even depth mismatch ────────────────────────────────────────────
    oe    = physical_feats["odd_even_ratio"]
    lo, hi = cfg.odd_even_ratio_limits
    if not (lo <= oe <= hi):
        v.passed_odd_even = False
        v.fp_reasons.append(
            f"Odd/even depth ratio={oe:.3f} outside [{lo},{hi}]"
        )
        fp += 0.20

    # ── 5. Too few transits to constrain period ───────────────────────────────
    if physical_feats["n_transits"] < 2:
        v.passed_transit_cnt = False
        v.fp_reasons.append("n_transits < 2 — period unconstrained")
        fp += 0.10

    v.fp_score = float(min(fp, 1.0))
    return v


vet = vet_candidate(physical_feats, tpf_collection, tls_result, cfg)

print("\n═══ VETTING REPORT ═══")
print(f"  Centroid shift   : {vet.centroid_shift_px:.4f}px  {'PASS' if vet.passed_centroid    else 'FAIL'}")
print(f"  Planet radius    : {physical_feats['rp_rjup']:.4f} RJup  {'PASS' if vet.passed_radius     else 'FAIL'}")
print(f"  Secondary eclipse: depth_ratio={physical_feats['secondary_depth']/(physical_feats['depth']+1e-10):.3f}  {'PASS' if vet.passed_secondary  else 'FAIL'}")
print(f"  Odd/even ratio   : {physical_feats['odd_even_ratio']:.4f}  {'PASS' if vet.passed_odd_even   else 'FAIL'}")
print(f"  Transit count    : {int(physical_feats['n_transits'])}  {'PASS' if vet.passed_transit_cnt else 'FAIL'}")
print(f"  FP Score         : {vet.fp_score:.3f}  (0=clean, 1=certain FP)")
if vet.fp_reasons:
    for r in vet.fp_reasons:
        print(f"  ⚠️  {r}")
else:
    print("  ✅ No FP flags raised.")


In [ ]:
# ── Phase 7: Feature Assembly + Ensemble Classifier ──────────────────────────

PHYS_KEYS = [
    "period", "depth", "duration_h", "sde", "snr", "rp_rjup",
    "odd_depth", "even_depth", "odd_even_ratio", "secondary_depth",
    "n_transits", "transit_count_quality", "ae_anomaly_frac",
    "rstar", "teff", "bls_sde", "tls_snr", "tls_odd_even",
    "tls_even_odd", "tls_depth_mean_odd", "tls_depth_mean_even",
    "period_uncertainty", "empty_transit_count", "in_transit_count",
]

# Vetting adds 5 more features
VET_KEYS = [
    "centroid_shift_px", "fp_score",
    "passed_centroid", "passed_radius", "passed_secondary",
]


def assemble_feature_vector(physical_feats: Dict,
                             vet: VettingResult,
                             cnn_vec: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """
    Returns:
        full_vec:  CNN latent + physics + vetting  (for CNN-head or combined model)
        phys_vec:  physics + vetting only           (for XGBoost / RF)
    """
    phys_arr = np.array([physical_feats[k] for k in PHYS_KEYS], dtype=np.float32)
    vet_arr  = np.array([
        vet.centroid_shift_px,
        vet.fp_score,
        float(vet.passed_centroid),
        float(vet.passed_radius),
        float(vet.passed_secondary),
    ], dtype=np.float32)

    phys_vec = np.concatenate([phys_arr, vet_arr])
    full_vec = np.concatenate([cnn_vec.astype(np.float32), phys_vec])
    return full_vec, phys_vec


# ── Model builders ────────────────────────────────────────────────────────────
def build_xgb() -> xgb.XGBClassifier:
    return xgb.XGBClassifier(
        n_estimators      = 600,
        max_depth         = 6,
        learning_rate     = 0.04,
        subsample         = 0.8,
        colsample_bytree  = 0.8,
        min_child_weight  = 3,
        gamma             = 0.1,
        reg_alpha         = 0.05,
        reg_lambda        = 1.0,
        eval_metric       = "logloss",
        random_state      = cfg.random_seed,
        n_jobs            = cfg.n_jobs,
        tree_method       = "hist",    # GPU-aware; falls back to CPU automatically
    )


def build_rf() -> RandomForestClassifier:
    return RandomForestClassifier(
        n_estimators     = 400,
        max_depth        = None,
        min_samples_leaf = 2,
        class_weight     = "balanced",
        random_state     = cfg.random_seed,
        n_jobs           = cfg.n_jobs,
    )


class CNNClassificationHead(nn.Module):
    """MLP head applied to CNN latent vector for planet/FP classification."""
    def __init__(self, latent_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(128, 32),         nn.GELU(),
            nn.Linear(32, 2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


full_feat_vec, phys_feat_vec = assemble_feature_vector(physical_feats, vet, cnn_vector)
print(f"Feature vector sizes:")
print(f"  Full (CNN+phys+vet)  : {full_feat_vec.shape[0]}")
print(f"  Physics+vet only     : {phys_feat_vec.shape[0]}")
print(f"  CNN latent           : {cnn_vector.shape[0]}")


In [ ]:
# ── Phase 7B: Training on Labelled Catalog ────────────────────────────────────
# Feed this function a labelled dataset such as:
#   - Kepler DR25 TCE catalog (KOI table, labelled PC/FP)
#   - TESS TOI catalog  (TFOP vetting dispositions)
#
# X_phys  : (N, len(PHYS_KEYS) + len(VET_KEYS)) — physics+vetting features
# X_cnn   : (N, cnn_latent_dim)                  — CNN latent vectors
# y       : (N,)  1=planet candidate, 0=false positive

def train_ensemble(
    X_phys: np.ndarray,
    X_cnn:  np.ndarray,
    y:      np.ndarray,
    cfg:    PipelineConfig,
) -> Dict:
    """
    Trains XGBoost, Random Forest, and CNN head.
    Calibrates XGB + RF probabilities with isotonic regression.
    Evaluates on a held-out 15 % validation set.
    Returns a dict of trained models and scalers.
    """
    X_tr_p, X_val_p, X_tr_c, X_val_c, y_tr, y_val = train_test_split(
        X_phys, X_cnn, y,
        test_size  = 0.15,
        stratify   = y,
        random_state = cfg.random_seed,
    )

    # ── Scale physics features (fit on TRAIN only — no leakage) ──────────────
    scaler_p   = StandardScaler().fit(X_tr_p)
    X_tr_p_s   = scaler_p.transform(X_tr_p)
    X_val_p_s  = scaler_p.transform(X_val_p)

    scaler_c   = StandardScaler().fit(X_tr_c)
    X_tr_c_s   = scaler_c.transform(X_tr_c).astype(np.float32)
    X_val_c_s  = scaler_c.transform(X_val_c).astype(np.float32)

    n_neg, n_pos = (y_tr == 0).sum(), (y_tr == 1).sum()
    scale_pos = n_neg / (n_pos + 1e-8)

    # ── XGBoost ───────────────────────────────────────────────────────────────
    logger.info("Training XGBoost ...")
    xgb_clf = build_xgb()
    xgb_clf.set_params(scale_pos_weight=scale_pos,
                       early_stopping_rounds=40)
    xgb_clf.fit(X_tr_p_s, y_tr,
                eval_set=[(X_val_p_s, y_val)],
                verbose=50)
    xgb_cal = CalibratedClassifierCV(xgb_clf, method="isotonic", cv="prefit")
    xgb_cal.fit(X_val_p_s, y_val)

    # ── Random Forest ─────────────────────────────────────────────────────────
    logger.info("Training Random Forest ...")
    rf_clf = build_rf()
    rf_clf.fit(X_tr_p_s, y_tr)
    rf_cal = CalibratedClassifierCV(rf_clf, method="isotonic", cv="prefit")
    rf_cal.fit(X_val_p_s, y_val)

    # ── CNN head ──────────────────────────────────────────────────────────────
    logger.info("Training CNN head ...")
    cnn_head = CNNClassificationHead(cfg.cnn_latent_dim).to(DEVICE)
    pos_wt   = torch.tensor([1.0, scale_pos], dtype=torch.float32).to(DEVICE)
    crit     = nn.CrossEntropyLoss(weight=pos_wt)
    opt      = optim.AdamW(cnn_head.parameters(), lr=1e-3, weight_decay=1e-3)
    sched    = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=60)

    t_X = torch.tensor(X_tr_c_s).to(DEVICE)
    t_y = torch.tensor(y_tr).long().to(DEVICE)
    dl  = DataLoader(TensorDataset(t_X, t_y), batch_size=32, shuffle=True)

    cnn_head.train()
    for ep in tqdm(range(60), desc="CNN head"):
        for xb, yb in dl:
            logits = cnn_head(xb)
            loss   = crit(logits, yb)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()

    # ── Evaluate all three ────────────────────────────────────────────────────
    xgb_auc = roc_auc_score(y_val, xgb_cal.predict_proba(X_val_p_s)[:, 1])
    rf_auc  = roc_auc_score(y_val, rf_cal.predict_proba(X_val_p_s)[:, 1])
    cnn_head.eval()
    with torch.no_grad():
        cnn_probs = torch.softmax(
            cnn_head(torch.tensor(X_val_c_s).to(DEVICE)), dim=1
        )[:, 1].cpu().numpy()
    cnn_auc = roc_auc_score(y_val, cnn_probs)

    logger.info(
        f"Val AUC — XGB={xgb_auc:.4f}  RF={rf_auc:.4f}  CNN={cnn_auc:.4f}"
    )
    print(classification_report(y_val,
                                  xgb_cal.predict(X_val_p_s),
                                  target_names=["FP","Planet"]))
    return {
        "xgb":      xgb_cal,
        "rf":       rf_cal,
        "cnn_head": cnn_head,
        "scaler_p": scaler_p,
        "scaler_c": scaler_c,
        "val_auc":  {"xgb": xgb_auc, "rf": rf_auc, "cnn": cnn_auc},
    }


print("train_ensemble() defined.")
print("Usage: models = train_ensemble(X_phys, X_cnn, y, cfg)")


In [ ]:
# ── Phase 8: Consensus Voting & Final Decision ────────────────────────────────

@dataclass
class PredictionResult:
    cnn_prob:        float = 0.0
    xgb_prob:        float = 0.0
    rf_prob:         float = 0.0
    ensemble_prob:   float = 0.0
    fp_adjusted_prob:float = 0.0
    verdict:         str   = "UNKNOWN"
    confidence:      str   = "LOW"
    explanation:     str   = ""


def ensemble_predict(
    cnn_prob: float,
    xgb_prob: float,
    rf_prob:  float,
    vet:      VettingResult,
    cfg:      PipelineConfig,
) -> PredictionResult:
    """
    Weighted consensus vote with hard vetting overrides.

    Hard overrides (physique impossibilities) take precedence over soft votes.
    Continuous FP score continuously penalises the ensemble probability.
    """
    res = PredictionResult(cnn_prob=cnn_prob, xgb_prob=xgb_prob, rf_prob=rf_prob)

    w = cfg.ensemble_weights
    ensemble_p = (w["cnn"] * cnn_prob + w["xgb"] * xgb_prob + w["rf"] * rf_prob)
    res.ensemble_prob    = ensemble_p
    res.fp_adjusted_prob = float(np.clip(ensemble_p * (1.0 - vet.fp_score), 0, 1))

    # ── Hard overrides ────────────────────────────────────────────────────────
    if not vet.passed_radius:
        res.verdict     = "FALSE_POSITIVE"
        res.confidence  = "HIGH"
        res.explanation = ("Rp > 2 RJup — this is an eclipsing binary, not a planet.")
        return res
    if not vet.passed_centroid:
        res.verdict     = "FALSE_POSITIVE"
        res.confidence  = "HIGH"
        res.explanation = ("Centroid shifts during transit — background eclipsing binary.")
        return res
    if not vet.passed_secondary:
        res.verdict     = "FALSE_POSITIVE"
        res.confidence  = "MEDIUM"
        res.explanation = ("Secondary eclipse at phase=0.5 — eclipsing binary system.")
        return res

    # ── Soft voting ───────────────────────────────────────────────────────────
    p       = res.fp_adjusted_prob
    yes_xgb = xgb_prob >= 0.5
    yes_rf  = rf_prob  >= 0.5
    yes_cnn = cnn_prob >= 0.5
    n_yes   = int(yes_xgb) + int(yes_rf) + int(yes_cnn)

    if n_yes == 3 and p >= cfg.high_confidence_threshold:
        res.verdict, res.confidence = "PLANET_CANDIDATE", "HIGH"
        res.explanation = f"All three models agree: planet (p={p:.3f}). All vetting checks passed."
    elif n_yes == 3:
        res.verdict, res.confidence = "PLANET_CANDIDATE", "MEDIUM"
        res.explanation = f"All models vote planet, but FP-adjusted probability is modest (p={p:.3f})."
    elif n_yes == 2:
        res.verdict, res.confidence = "WEAK_SIGNAL", "LOW"
        res.explanation = f"Majority vote: planet ({n_yes}/3 models). Manual review recommended."
    elif n_yes == 0 and p < 0.30:
        res.verdict, res.confidence = "NON_DETECTION", "HIGH"
        res.explanation = f"All models agree: no planet (p={p:.3f})."
    else:
        res.verdict, res.confidence = "AMBIGUOUS", "LOW"
        res.explanation = f"Mixed votes ({n_yes}/3 planet). Expert review required."

    return res


# ── Demo with heuristic probabilities (no trained model loaded yet) ───────────
# When trained models are available, replace these with real predict_proba calls
def heuristic_prob(physical_feats: Dict, vet: VettingResult) -> float:
    """Heuristic fallback probability when no trained classifier is available."""
    sde_norm = float(np.clip(physical_feats["sde"] / 15.0, 0, 1))
    return float(sde_norm * (1.0 - vet.fp_score))


h_prob = heuristic_prob(physical_feats, vet)
prediction = ensemble_predict(h_prob, h_prob, h_prob, vet, cfg)

print("\n╔══════════════════════════════════════════════════╗")
print(f"║  VERDICT     : {prediction.verdict:<34}║")
print(f"║  CONFIDENCE  : {prediction.confidence:<34}║")
print(f"║  P(planet)   : {prediction.fp_adjusted_prob:<34.4f}║")
print("╚══════════════════════════════════════════════════╝")
print(f"\n  {prediction.explanation}")
print(f"\n  CNN={prediction.cnn_prob:.3f}  XGB={prediction.xgb_prob:.3f}  RF={prediction.rf_prob:.3f}")
print(f"  Ensemble={prediction.ensemble_prob:.3f}  FP-adjusted={prediction.fp_adjusted_prob:.3f}")


In [ ]:
# ── Phase 9: Six-Panel Diagnostic Visualisation ───────────────────────────────

def plot_diagnostics(
    time_raw, flux_raw,
    time_det, flux_det, stellar_trend,
    recon_error, ae_anomaly, ae_threshold,
    tls_result,
    phase_profile, phase_bins,
    physical_feats,
    vet, prediction,
    cfg: PipelineConfig,
    save_path: Optional[str] = None,
):
    """Six-panel diagnostic plot matching professional exoplanet pipeline style."""
    fig = plt.figure(figsize=(18, 20), facecolor="#0d1117")
    gs  = gridspec.GridSpec(4, 2, figure=fig, hspace=0.48, wspace=0.32)

    C_BG   = "#0d1117"
    C_CARD = "#161b22"
    C_GRID = "#21262d"
    C_TEXT = "#e6edf3"
    C_DIM  = "#8b949e"
    C_BLUE = "#58a6ff"
    C_RED  = "#f85149"
    C_GRN  = "#3fb950"
    C_YELL = "#d29922"

    def style(ax, title):
        ax.set_facecolor(C_CARD)
        ax.set_title(title, color=C_TEXT, fontsize=10.5, pad=7)
        ax.tick_params(colors=C_DIM, labelsize=8.5)
        for sp in ax.spines.values():
            sp.set_color(C_GRID)
        ax.xaxis.label.set_color(C_DIM)
        ax.yaxis.label.set_color(C_DIM)
        ax.grid(color=C_GRID, linewidth=0.5)

    # ── 1. Raw stitched LC ─────────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, :])
    ax1.scatter(time_raw, flux_raw, s=0.5, alpha=0.55, color=C_BLUE, rasterized=True)
    style(ax1, f"Raw Stitched Light Curve — {cfg.tic_id}")
    ax1.set_xlabel("BJD − 2457000"); ax1.set_ylabel("Norm. Flux")

    # ── 2. Detrended + LSTM-AE anomaly flags ──────────────────────────────────
    ax2 = fig.add_subplot(gs[1, :])
    ax2.scatter(time_det, flux_det, s=0.4, alpha=0.45, color=C_BLUE,
                rasterized=True, label="Detrended")
    if ae_anomaly.sum() > 0:
        ax2.scatter(time_det[ae_anomaly], flux_det[ae_anomaly],
                    s=5, color=C_RED, alpha=0.85, zorder=5,
                    label="LSTM-AE anomaly")
    if tls_result is not None:
        for t0 in tls_result.transit_times:
            ax2.axvline(t0, color=C_GRN, alpha=0.25, linewidth=0.9)
    style(ax2, "Detrended LC + LSTM-AE Anomaly Flags (green = TLS transit times)")
    ax2.set_xlabel("BJD − 2457000"); ax2.set_ylabel("Norm. Flux")
    ax2.legend(facecolor=C_CARD, labelcolor=C_TEXT, markerscale=4, fontsize=8)

    # ── 3. LSTM-AE reconstruction error ───────────────────────────────────────
    ax3 = fig.add_subplot(gs[2, 0])
    ax3.plot(time_det, recon_error, color=C_YELL, linewidth=0.55, alpha=0.9)
    ax3.axhline(ae_threshold, color=C_RED, linewidth=1.1, linestyle="--",
                label=f"P{cfg.ae_anomaly_percentile:.0f} threshold")
    style(ax3, "LSTM-AE Reconstruction Error")
    ax3.set_xlabel("BJD − 2457000"); ax3.set_ylabel("MSE")
    ax3.legend(facecolor=C_CARD, labelcolor=C_TEXT, fontsize=8)

    # ── 4. TLS periodogram ────────────────────────────────────────────────────
    ax4 = fig.add_subplot(gs[2, 1])
    if tls_result is not None:
        ax4.plot(tls_result.periods, tls_result.power,
                 color=C_BLUE, linewidth=0.65, alpha=0.9)
        ax4.axvline(tls_result.period, color=C_GRN, linewidth=1.5,
                    label=f"P = {tls_result.period:.4f} d")
        ax4.axhline(cfg.tls_sde_threshold, color=C_RED, linewidth=1.0,
                    linestyle="--", label=f"SDE = {cfg.tls_sde_threshold}")
        sde_label = f"TLS Periodogram (SDE = {tls_result.SDE:.2f})"
    else:
        ax4.text(0.5, 0.5, "TLS not run\n(BLS gate not triggered)",
                 ha="center", va="center", color=C_DIM, transform=ax4.transAxes)
        sde_label = "TLS Periodogram"
    style(ax4, sde_label)
    ax4.set_xlabel("Period (days)"); ax4.set_ylabel("SDE")
    ax4.legend(facecolor=C_CARD, labelcolor=C_TEXT, fontsize=8)

    # ── 5. Phase-folded transit ────────────────────────────────────────────────
    ax5 = fig.add_subplot(gs[3, 0])
    ax5.plot(phase_bins, phase_profile, color=C_BLUE, linewidth=1.4)
    if tls_result is not None:
        ax5.axhline(1.0 - physical_feats["depth"], color=C_RED, linewidth=0.9,
                    linestyle="--",
                    label=f"depth = {physical_feats['depth']:.5f}")
    style(ax5, "Phase-Folded Transit (CNN input)")
    ax5.set_xlabel("Phase"); ax5.set_ylabel("Norm. Flux")
    ax5.legend(facecolor=C_CARD, labelcolor=C_TEXT, fontsize=8)

    # ── 6. Results panel ──────────────────────────────────────────────────────
    ax6 = fig.add_subplot(gs[3, 1])
    ax6.set_facecolor(C_CARD); ax6.axis("off")
    ax6.set_title("Prediction Summary", color=C_TEXT, fontsize=10.5, pad=7)
    for sp in ax6.spines.values():
        sp.set_color(C_GRID)

    v_color = C_GRN if "PLANET" in prediction.verdict else C_RED

    rows = [
        ("VERDICT",        prediction.verdict,            v_color),
        ("Confidence",     prediction.confidence,         C_TEXT),
        ("P(planet)",      f"{prediction.fp_adjusted_prob:.4f}", C_TEXT),
        ("───────────────", "",                            C_GRID),
        ("Period",         f"{physical_feats['period']:.5f} d",  C_TEXT),
        ("Depth",          f"{physical_feats['depth']:.5f}",     C_TEXT),
        ("Duration",       f"{physical_feats['duration_h']:.2f} h", C_TEXT),
        ("Rp",             f"{physical_feats['rp_rjup']:.3f} RJup", C_TEXT),
        ("SDE",            f"{physical_feats['sde']:.2f}",        C_TEXT),
        ("SNR",            f"{physical_feats['snr']:.2f}",        C_TEXT),
        ("───────────────", "",                            C_GRID),
        ("Centroid",       f"{'PASS' if vet.passed_centroid  else 'FAIL'} ({vet.centroid_shift_px:.3f}px)", C_GRN if vet.passed_centroid  else C_RED),
        ("Radius",         f"{'PASS' if vet.passed_radius    else 'FAIL'} ({physical_feats['rp_rjup']:.2f} RJup)", C_GRN if vet.passed_radius    else C_RED),
        ("Secondary",      f"{'PASS' if vet.passed_secondary else 'FAIL'}", C_GRN if vet.passed_secondary else C_RED),
        ("Odd/Even",       f"{'PASS' if vet.passed_odd_even  else 'FAIL'} (r={physical_feats['odd_even_ratio']:.3f})", C_GRN if vet.passed_odd_even  else C_RED),
        ("FP Score",       f"{vet.fp_score:.3f}",          C_GRN if vet.fp_score < 0.2 else C_RED),
        ("───────────────", "",                            C_GRID),
        ("CNN prob",       f"{prediction.cnn_prob:.4f}",  C_TEXT),
        ("XGB prob",       f"{prediction.xgb_prob:.4f}",  C_TEXT),
        ("RF  prob",       f"{prediction.rf_prob:.4f}",   C_TEXT),
    ]

    for i, (label, val, col) in enumerate(rows):
        y = 0.97 - i * 0.047
        is_sep = "───" in label
        ax6.text(0.03, y, label, color=C_GRID if is_sep else C_DIM,
                 fontsize=8.0, transform=ax6.transAxes, va="top")
        ax6.text(0.50, y, val, color=col, fontsize=8.0,
                 transform=ax6.transAxes, va="top", fontweight="bold")

    fig.suptitle(
        f"Exoplanet Detection Pipeline v4.0 — {cfg.tic_id}",
        color=C_TEXT, fontsize=13, fontweight="bold", y=0.997,
    )

    out = save_path or f"diagnostic_{cfg.tic_id.replace(' ', '_')}.png"
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=C_BG)
    plt.show()
    print(f"Diagnostic saved → {out}")


phase_bins = np.linspace(-0.5, 0.5, cfg.cnn_phase_bins)

plot_diagnostics(
    time_raw, flux_raw,
    time_det, flux_det, stellar_trend,
    recon_error, ae_anomaly, ae_threshold,
    tls_result,
    phase_profile, phase_bins,
    physical_feats,
    vet, prediction,
    cfg,
)


In [ ]:
# ── Phase 10: run_pipeline() — Single-Target & Batch Wrapper ─────────────────

def run_pipeline(
    tic_id:         str,
    cfg:            Optional[PipelineConfig] = None,
    trained_models: Optional[Dict]          = None,
    plot:           bool                    = True,
    save_plot:      bool                    = True,
) -> Dict:
    """
    End-to-end pipeline for a single TIC target.

    Args:
        tic_id:         e.g. 'TIC 260647166'
        cfg:            PipelineConfig override (if None, uses defaults for tic_id)
        trained_models: output of train_ensemble() — uses heuristic fallback if None
        plot:           generate diagnostic figure
        save_plot:      save figure to PNG

    Returns:
        dict with keys: tic_id, tls_result, physical_features,
                        vetting, prediction, stellar_params, bls_sde, n_sectors
    """
    if cfg is None:
        cfg = PipelineConfig(tic_id=tic_id)
    else:
        cfg.tic_id = tic_id

    logger.info("=" * 60)
    logger.info(f"Pipeline start: {tic_id}")
    logger.info("=" * 60)

    # ── 1. Ingest ─────────────────────────────────────────────────────────────
    lc_col, tpf_col, stellar = ingest_all(cfg)
    n_sectors = len(lc_col)

    # ── 2. Quality filter + CBV + Stitch ──────────────────────────────────────
    lc_s = build_stitched_lc(lc_col, cfg)
    t_raw, f_raw = lc_s.time.value, lc_s.flux.value
    f_err_raw    = lc_s.flux_err.value

    # ── 3. BLS pre-scan ───────────────────────────────────────────────────────
    b_p, b_t0, _, b_sde = bls_prescan(t_raw, f_raw, cfg)
    do_tls = b_sde >= cfg.bls_sde_threshold

    # ── 4. Wotan detrending ───────────────────────────────────────────────────
    t_mask = (build_transit_mask(t_raw, b_p, b_t0, cfg.transit_mask_duration_h)
              if do_tls else np.zeros(len(t_raw), dtype=bool))
    f_flat, trend = detrend_wotan(t_raw, f_raw, t_mask, cfg)
    valid  = np.isfinite(f_flat)
    t_det  = t_raw[valid]; f_det = f_flat[valid]
    e_det  = f_err_raw[valid]; m_det = t_mask[valid]

    # ── 5. LSTM-AE ────────────────────────────────────────────────────────────
    q_mask = ~m_det
    ae     = train_lstm_ae(t_det[q_mask], f_det[q_mask], cfg)
    rerr   = compute_recon_error(ae, t_det, f_det, cfg)
    ae_thr = np.percentile(rerr, cfg.ae_anomaly_percentile)
    ae_an  = rerr > ae_thr

    # ── 6. TLS ────────────────────────────────────────────────────────────────
    tls_res = None
    if do_tls:
        try:
            ab, R_s, M_s = catalog_info(TIC_ID=int(tic_id.replace("TIC ", "")))
        except Exception:
            ab, R_s, M_s = [0.4, 0.3], stellar.get("radius_sun", 1.0), 1.0
        if stellar.get("source") == "GaiaDR3":
            R_s = stellar["radius_sun"]
        t_c, f_c = cleaned_array(t_det, f_det)
        tls_mod  = transitleastsquares(t_c, f_c)
        tls_res  = tls_mod.power(
            R_star=float(R_s), M_star=float(M_s), u=list(ab),
            period_min=cfg.tls_min_period, period_max=cfg.tls_max_period,
            oversampling_factor=cfg.tls_oversampling,
            duration_grid_step=cfg.tls_duration_grid_step,
            use_threads=4, show_progress_bar=False,
        )
        if tls_res.SDE < cfg.tls_sde_threshold:
            tls_res = None

    # ── 7. Features ───────────────────────────────────────────────────────────
    if tls_res is not None:
        pp = phase_fold_profile(t_det, f_det, tls_res.period, tls_res.T0,
                                 cfg.cnn_phase_bins)
    else:
        pp = np.ones(cfg.cnn_phase_bins, dtype=np.float32)

    cnn_m = trained_models.get("cnn_encoder") if trained_models else None
    cv    = extract_cnn_vector(pp, cfg, model=cnn_m)
    pf    = extract_physical_features(t_det, f_det, tls_res, stellar, ae_an, b_sde)

    # ── 8. Vetting ────────────────────────────────────────────────────────────
    v = vet_candidate(pf, tpf_col, tls_res, cfg)

    # ── 9. Classification ─────────────────────────────────────────────────────
    if trained_models is not None:
        _, pv = assemble_feature_vector(pf, v, cv)
        sc_p  = trained_models["scaler_p"].transform(pv.reshape(1, -1))
        sc_c  = trained_models["scaler_c"].transform(cv.reshape(1, -1)).astype(np.float32)

        xgb_p = float(trained_models["xgb"].predict_proba(sc_p)[0, 1])
        rf_p  = float(trained_models["rf"].predict_proba(sc_p)[0, 1])
        trained_models["cnn_head"].eval()
        with torch.no_grad():
            cnn_p = float(torch.softmax(
                trained_models["cnn_head"](torch.tensor(sc_c).to(DEVICE)), dim=1
            )[0, 1])
    else:
        h = heuristic_prob(pf, v)
        xgb_p = cnn_p = rf_p = h

    pred = ensemble_predict(cnn_p, xgb_p, rf_p, v, cfg)

    # ── 10. Plot ──────────────────────────────────────────────────────────────
    if plot:
        save = f"diagnostic_{tic_id.replace(' ', '_')}.png" if save_plot else None
        plot_diagnostics(
            t_raw, f_raw, t_det, f_det, trend,
            rerr, ae_an, ae_thr, tls_res,
            pp, np.linspace(-0.5, 0.5, cfg.cnn_phase_bins),
            pf, v, pred, cfg,
            save_path=save,
        )

    return {
        "tic_id":             tic_id,
        "tls_result":         tls_res,
        "physical_features":  pf,
        "vetting":            v,
        "prediction":         pred,
        "stellar_params":     stellar,
        "bls_sde":            b_sde,
        "n_sectors":          n_sectors,
    }


# ── Run on the configured target ──────────────────────────────────────────────
result = run_pipeline(cfg.tic_id, cfg=cfg, trained_models=None, plot=True)
print("\nFinal verdict:", result["prediction"].verdict)


In [ ]:
# ── Phase 11: Batch Search Over Multiple TIC IDs ─────────────────────────────

def batch_search(
    tic_list:       List[str],
    cfg_template:   PipelineConfig,
    trained_models: Optional[Dict] = None,
    plot:           bool           = False,
) -> pd.DataFrame:
    """
    Run the full pipeline over a list of TIC IDs and return a summary DataFrame.
    Results are sorted by descending FP-adjusted probability.
    """
    records = []
    for tic in tic_list:
        try:
            r    = run_pipeline(tic, cfg=PipelineConfig(tic_id=tic),
                                trained_models=trained_models, plot=plot)
            pred = r["prediction"]
            pf   = r["physical_features"]
            v    = r["vetting"]
            records.append({
                "tic_id":        tic,
                "verdict":       pred.verdict,
                "confidence":    pred.confidence,
                "p_planet":      round(pred.fp_adjusted_prob, 4),
                "ensemble_prob": round(pred.ensemble_prob,    4),
                "bls_sde":       round(r["bls_sde"],          3),
                "tls_ran":       r["tls_result"] is not None,
                "period_d":      round(pf.get("period",  0), 5),
                "depth":         round(pf.get("depth",   0), 6),
                "rp_rjup":       round(pf.get("rp_rjup", 0), 3),
                "n_transits":    int(pf.get("n_transits", 0)),
                "fp_score":      round(v.fp_score, 3),
                "n_sectors":     r["n_sectors"],
            })
        except Exception as exc:
            logger.warning(f"Pipeline failed for {tic}: {exc}")
            records.append({"tic_id": tic, "verdict": "PIPELINE_ERROR",
                            "p_planet": 0.0})

    df = pd.DataFrame(records).sort_values("p_planet", ascending=False)
    df.to_csv("batch_results.csv", index=False)
    print(f"\nBatch complete — {len(df)} targets processed.")
    print(df[["tic_id","verdict","confidence","p_planet","bls_sde","period_d","rp_rjup"]].to_string(index=False))
    return df


# ── Example batch run ─────────────────────────────────────────────────────────
candidate_tics = [
    "TIC 307210830",   # known TESS planet host
    "TIC 402026209",
    "TIC 424007903",
    "TIC 271971130",
    "TIC 260647166",   # reference non-detection
]

# Uncomment to run:
# df_results = batch_search(candidate_tics, cfg_template=PipelineConfig())
# display(df_results)
print("batch_search() defined. Uncomment last two lines to run.")


## ✅ Pipeline v4.0 — What Was Fixed & Improved

| # | Issue in v3 | Fix in v4 |
|---|-------------|-----------|
| 1 | `%pip install` is not valid Python (SyntaxError in cell 0) | Replaced with `subprocess.check_call` — valid pure Python |
| 2 | `break_tolerance=0.5` duplicated kwarg (SyntaxError) | Removed duplicate; single `break_tolerance=0.5` |
| 3 | `train_lstm_ae` called `quiet_flux` but it was passed as arg named `flux_quiet` | Fixed: now correctly receives and uses `time_quiet, flux_quiet` |
| 4 | LSTM-AE `__init__` passed `dropout` to LSTM when `num_layers=1` (PyTorch raises warning/error) | `drop_arg = dropout if layers > 1 else 0.0` guards this correctly |
| 5 | `f"Starting pipeline for {tic_id}"` nested quotes broke f-string in `run_pipeline` | All f-strings with nested quotes rewritten to avoid nesting |
| 6 | `xgb.XGBClassifier(use_label_encoder=False)` — deprecated kwarg since XGB 1.6 | Removed; XGB no longer uses label encoder by default |
| 7 | `ingest_all = ingest_all_robust` monkey-patches module-level name silently | Robust Gaia fallback built directly into `ingest_all` from the start |
| 8 | `_fetch_gaia` assumed Gaia masked arrays respond to `row['col'].mask[0] is False` | Replaced with `safe_col()` that handles masked, nan, None uniformly |
| 9 | CNN encoder used ReLU; smoother signal benefits from GELU | Replaced all ReLU with GELU in CNN layers |
| 10 | `ingest_all` re-defined in cell 20 — late override breaks earlier cells | Removed override; single `ingest_all` with built-in robustness |
| 11 | No `torch.no_grad()` guard on inference | All inference paths use `@torch.no_grad()` or `with torch.no_grad()` |
| 12 | Physical feature dict could contain NaN/masked floats — XGBoost fails silently | `sf()` helper coerces all values to finite float with explicit fallbacks |
| 13 | BLS used `astropy.time.Time` import that wasn't needed | Removed unused import; `u.day` alone is sufficient |
| 14 | CNN encoder training was a stub with no training loop | Full supervised `train_cnn_encoder()` added with class-weighted loss, cosine LR, early AUC tracking |
| 15 | `Tuple` used as type hint without import | Added `Tuple` to `from typing import` imports |
